### News Sentiment (NewsAPI + Finnhub, last 10 days)

- **Output**: `Reports/news_cleaned_df.csv` (relevant non-neutral articles, used by the app's AI news summaries),
  `Reports/weighted_sentiment.csv` (SentimentScore −10..+10 for **every** tracked symbol; 0 = no relevant news),
  `Reports/sentiment_history.csv` (appended each online run: date, symbol, score, article counts).
- **Modes** (env `PIPELINE_SENTIMENT_MODE`, default `online` when run by hand; `run_all.py` sets it):
  - `online` — fetch news for all symbols (≈97 NewsAPI + 97 Finnhub calls (96 stocks + QQQ; NewsAPI free limit 100/day)), score with FinBERT, write outputs.
  - `offline` — no network: re-apply the relevance filter to the cached `news_cleaned_df.csv` and recompute the scores
    (idempotent; used by `run_all.py` in quick mode and whenever NewsAPI already ran in the last 24 h).
- **Quota guard**: run by hand in `online` mode, the notebook refuses to call NewsAPI again within 24 h of the last run
  (read from `Reports/run_state.json` / `sentiment_history.csv`); set `PIPELINE_FORCE_NEWS=1` to override.
  Every NewsAPI/Finnhub call is counted and printed at the end (and reported in the `run_all.py` summary).
  - `sample` — fetch 1–2 symbols (`PIPELINE_SAMPLE_SYMBOLS`, default `NVDA`) end to end and write only to `Reports/cache/`.
- Relevance: `is_relevant` in the **Relevance filter** section (ticker forms or distinctive company names/aliases).

### Setup
Mode (online / offline / sample), symbols, API keys from `.env` (never printed) and file paths.

In [ ]:
import json
import os
import time
from datetime import datetime, timedelta, timezone

import pandas as pd
from dotenv import load_dotenv

import sector_mapping
from sector_mapping import stock_symbols

load_dotenv(os.path.join(sector_mapping.PROJECT_ROOT, ".env"))
MODE = os.getenv("PIPELINE_SENTIMENT_MODE", "online").lower()
assert MODE in {"online", "offline", "sample"}, MODE
REPORTS_DIR = sector_mapping.REPORTS_DIR
CACHE_DIR = os.path.join(REPORTS_DIR, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)
NEWS_CSV = os.path.join(REPORTS_DIR, "news_cleaned_df.csv")
SCORE_CSV = os.path.join(REPORTS_DIR, "weighted_sentiment.csv")
HISTORY_CSV = os.path.join(REPORTS_DIR, "sentiment_history.csv")
SYMBOLS = [s.strip().upper() for s in os.getenv("PIPELINE_SAMPLE_SYMBOLS", "NVDA").split(",")][:2] if MODE == "sample" else stock_symbols

END = datetime.now(timezone.utc).date()
START = END - timedelta(days=10)
POSITIVE_CUTOFF, NEGATIVE_CUTOFF = 0.75, 0.65
PRIOR_ARTICLES = 5
print(f"Mode: {MODE} | symbols: {len(SYMBOLS)} | window {START} → {END}")


def with_retries(fn, tries=3, base_wait=2):
    """Call fn(); retry transient failures with exponential backoff, re-raise the last error."""
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            if attempt == tries - 1 or "rateLimited" in str(e) or "maximumResultsReached" in str(e):
                raise
            time.sleep(base_wait * 2 ** attempt)

### Relevance filter (is this article really about the stock?)
An article counts for a symbol when ANY of these holds: an explicit ticker form (`$TICK`, `(TICK)`, `NASDAQ:TICK`,
`TICK stock/shares`); the bare upper-case ticker, unless it is also a common word (`AMBIGUOUS_TICKERS`); or a distinctive
company name / brand alias as whole words (`ALIASES`; names in `CASE_SENSITIVE_ALIASES` must match the case; a tuple means
all its words must appear). The old rule matched the first word of the company name, so "advanced" (AMD) or "super" (SMCI)
matched unrelated news. **When you add a stock to the universe, add its aliases here.**

In [ ]:
import re

from sector_mapping import symbol_name

SUFFIXES = {"inc", "inc.", "corp", "corp.", "corporation", "co", "co.", ".inc", "ltd", "ltd.",
            "plc", "group", "holding", "holdings", "company", "companies", "nv", "n.v."}

AMBIGUOUS_TICKERS = {"U", "UI", "IT", "APP", "NOW", "ARM", "TEAM", "HOOD", "FIG", "CRM", "COIN", "SNOW", "SHOP",
                     "DASH", "MU", "ZS", "TEM", "HAL", "UBER", "META", "MARA", "HIMS", "SOFI", "A",
                     "APA", "FANG", "URI",   # 2026-09-24 U91 additions: APA (psych. assoc.), FANG (= FAANG stocks), URI (web term)
                     "CLS", "LITE"}          # 2026-09-24 U96 additions: CLS (CLS Bank, Cumulative Layout Shift), LITE (the word "lite")

ALIASES = {
    "AAPL": ["Apple"], "ADBE": ["Adobe"], "AFRM": ["Affirm Holdings", "Affirm"], "AMZN": ["Amazon"],
    "ANET": ["Arista Networks", "Arista"], "AMD": ["Advanced Micro Devices"], "APP": ["AppLovin"],
    "AVAV": ["AeroVironment"], "AVGO": ["Broadcom"], "BIIB": ["Biogen"], "BKR": ["Baker Hughes"],
    "CDNS": ["Cadence Design"], "COIN": ["Coinbase"], "CRM": ["Salesforce"], "CRWV": ["CoreWeave"],
    "CVLT": ["Commvault"], "DASH": ["DoorDash"], "ENPH": ["Enphase"], "FIG": ["Figma"], "FTNT": ["Fortinet"],
    "GOOGL": ["Alphabet", "Google"], "GTLB": ["GitLab"], "HAL": ["Halliburton"],
    "HIMS": ["Hims & Hers", "Hims and Hers", "Hims&Hers"], "HOOD": ["Robinhood"], "INTU": ["Intuit", "TurboTax"],
    "IREN": ["Iris Energy", "IREN Limited"], "LRCX": ["Lam Research"], "MCK": ["McKesson"], "MRK": ["Merck"],
    "META": ["Meta Platforms", "Meta", "Facebook"], "MRVL": ["Marvell"], "MSFT": ["Microsoft"], "MU": ["Micron"],
    "NFLX": ["Netflix"], "NOW": ["ServiceNow"], "NVDA": ["Nvidia"], "ORCL": ["Oracle"], "PANW": ["Palo Alto Networks"],
    "RCL": ["Royal Caribbean"], "REGN": ["Regeneron"], "SLB": ["Schlumberger"], "SNOW": ["Snowflake"],
    "SOFI": ["SoFi"], "TEAM": ["Atlassian"], "TSLA": ["Tesla"], "UBER": ["Uber"], "UI": ["Ubiquiti"],
    "UNH": ["UnitedHealth"], "UPST": ["Upstart"], "VEEV": ["Veeva"], "VRT": ["Vertiv"], "ZS": ["Zscaler"],
    "UMAC": ["Unusual Machines"], "NOC": ["Northrop Grumman", "Northrop"], "MDB": ["MongoDB"],
    "LMT": ["Lockheed Martin", "Lockheed"], "U": ["Unity Software", "Unity Technologies", ("Unity", "stock")], "TWLO": ["Twilio"],
    "CRCL": ["Circle Internet", ("Circle", "USDC"), ("Circle", "stablecoin"), ("Circle", "stablecoins")], "ACHR": ["Archer Aviation"],
    "ALAB": ["Astera Labs"], "APLD": ["Applied Digital"], "ARM": ["Arm Holdings", "Arm"], "ASTS": ["AST SpaceMobile"],
    "CIFR": ["Cipher Mining", "Cipher Digital"], "CVNA": ["Carvana"], "IONQ": ["IonQ"], "JOBY": ["Joby Aviation", "Joby"],
    "MARA": ["MARA Holdings", "Marathon Digital"], "MSTR": ["MicroStrategy", "Strategy Inc", "Michael Saylor", "Saylor"],
    "NVTS": ["Navitas Semiconductor", "Navitas"], "RDDT": ["Reddit"], "RKLB": ["Rocket Lab"], "SHOP": ["Shopify"],
    "SMCI": ["Super Micro", "Supermicro"], "SOUN": ["SoundHound"], "TEM": ["Tempus AI"], "QQQ": ["Invesco QQQ"],
    # U91 additions (2026-09-24)
    "APA": ["APA Corporation", "APA Corp", "Apache Corporation"], "OXY": ["Occidental Petroleum", "Occidental"],
    "TRGP": ["Targa Resources", "Targa"], "DVN": ["Devon Energy"], "FANG": ["Diamondback Energy"],
    "COF": ["Capital One"], "C": ["Citigroup", "Citibank", ("Citi", "bank")], "BE": ["Bloom Energy"], "BA": ["Boeing"],
    "URI": ["United Rentals"], "PH": ["Parker-Hannifin", "Parker Hannifin"], "FCX": ["Freeport-McMoRan", "Freeport McMoRan"],
    "LYB": ["LyondellBasell"],
    # U96 additions (2026-09-24)
    "CRDO": ["Credo Technology", ("Credo", "semiconductor"), ("Credo", "connectivity"), ("Credo", "AI")], "NBIS": ["Nebius"],
    "LITE": ["Lumentum"], "CLS": ["Celestica"], "RBRK": ["Rubrik"],
}
CASE_SENSITIVE_ALIASES = {"Meta", "Uber", "Affirm", "Joby", "Navitas", "Oracle", "Apple", "Amazon", "Tempus AI",
                          "Arm Holdings", "Arm", "Circle", "Unity", "Merck", "Northrop", "Lockheed", "Intuit",
                          "Occidental", "Targa", "Boeing", "Citi", "Credo"}


def clean_company_name(name):
    """Company name without punctuation and legal suffixes (Inc, Corp, ...)."""
    name = re.sub(r"\.com\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"[^\w\s&]", "", name)
    return " ".join(w for w in name.split() if w.lower() not in SUFFIXES).strip()


def aliases_for(symbol):
    """Curated aliases first, then the full cleaned company name (never a single generic first word)."""
    out = list(ALIASES.get(symbol, []))
    full = clean_company_name(symbol_name.get(symbol, ""))
    if full and full not in out and len(full.split()) >= 2:
        out.append(full)
    return out


def search_query(symbol):
    """NewsAPI query: the most distinctive alias as an exact phrase, OR the ticker when it is not a common word."""
    names = [a for a in aliases_for(symbol) if isinstance(a, str)]
    q = f'"{names[0]}"' if names else symbol
    if names and symbol not in AMBIGUOUS_TICKERS and len(symbol) >= 3:
        q += f" OR {symbol}"
    return q


def _has_phrase(phrase, text, case_sensitive):
    """True when the phrase appears as a whole word in the text."""
    flags = 0 if case_sensitive else re.IGNORECASE
    return re.search(r"(?<![\w$])" + re.escape(phrase) + r"(?!\w)", text, flags) is not None


def is_relevant(symbol, text):
    """True when the article text clearly refers to `symbol`."""
    if not isinstance(text, str) or not text:
        return False
    t = re.escape(symbol)
    strong = (rf"\${t}\b", rf"\({t}\)", rf"\b(?:NASDAQ|NYSE|Nasdaq|NYSEARCA|AMEX)\s*:\s*{t}\b", rf"\b{t}\s+(?:stock|shares)\b")
    if any(re.search(p, text) for p in strong):
        return True
    if symbol not in AMBIGUOUS_TICKERS and len(symbol) >= 3 and re.search(rf"(?<![\w$]){t}(?!\w)", text):
        return True
    for alias in aliases_for(symbol):
        terms = alias if isinstance(alias, tuple) else (alias,)
        if all(_has_phrase(a, text, a in CASE_SENSITIVE_ALIASES) for a in terms):
            return True
    return False

### Fetch news (online / sample only)

In [ ]:
articles, errors = [], []
CALLS = {"newsapi": 0, "finnhub": 0}


def counted(api, fn):
    """Count every request attempt (retries included) for the run summary."""
    def call():
        CALLS[api] += 1
        return fn()
    return call


STATE_FILE = os.path.join(REPORTS_DIR, "run_state.json")   # shared with run_all.py: NewsAPI at most once per 24 h
if MODE == "online" and os.getenv("PIPELINE_NEWS_APPROVED") != "1":      # run by hand (run_all.py applies the guard itself)
    state = json.load(open(STATE_FILE)) if os.path.exists(STATE_FILE) else {}
    last = state.get("last_news_at")
    if last and os.getenv("PIPELINE_FORCE_NEWS") != "1":
        gap_h = (datetime.now(timezone.utc) - datetime.fromisoformat(last)).total_seconds() / 3600
        if gap_h < 24:
            raise RuntimeError(f"NewsAPI already ran {gap_h:.1f} h ago (free limit 100 calls/day, one run = 97). "
                               "Set PIPELINE_FORCE_NEWS=1 to override, or use PIPELINE_SENTIMENT_MODE=offline.")
    state["last_news_at"] = datetime.now().astimezone().isoformat(timespec="seconds")
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=1, sort_keys=True)

if MODE in {"online", "sample"}:
    import finnhub
    from newsapi import NewsApiClient

    newsapi = NewsApiClient(api_key=os.getenv("NEWS_API_KEY"))
    finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))
    for symbol in SYMBOLS:
        try:
            res = with_retries(counted("newsapi", lambda: newsapi.get_everything(
                q=search_query(symbol), language="en", from_param=START.isoformat(), to=END.isoformat(), sort_by="publishedAt",
                page_size=25)))
        except Exception as e:
            errors.append(("newsapi", symbol, str(e)[:120]))
            if "rateLimited" in str(e):
                print("NewsAPI daily limit reached - continuing with Finnhub only")
                break
            continue
        articles += [{"symbol": symbol, "date": a.get("publishedAt"), "headline": a.get("title") or "",
                      "summary": a.get("description") or "", "source": (a.get("source") or {}).get("name", "")}
                     for a in res.get("articles", [])]
    for symbol in SYMBOLS:
        try:
            news = with_retries(counted("finnhub", lambda: finnhub_client.company_news(symbol, _from=START.isoformat(),
                                                                                       to=END.isoformat())))
        except Exception as e:
            errors.append(("finnhub", symbol, str(e)[:120]))
            time.sleep(1.1)  # keep the pace on errors too (Finnhub free tier: 60 calls/min)
            continue
        articles += [{"symbol": symbol, "date": datetime.fromtimestamp(n["datetime"], tz=timezone.utc).isoformat(),
                      "headline": n.get("headline") or "", "summary": n.get("summary") or "", "source": n.get("source", "")}
                     for n in news if START <= datetime.fromtimestamp(n["datetime"], tz=timezone.utc).date() <= END]
        time.sleep(1.1)  # Finnhub free tier: 60 calls/min
    print(f"Fetched {len(articles)} articles; {len(errors)} API errors")
    print(f"API calls: newsapi={CALLS['newsapi']} finnhub={CALLS['finnhub']}")
    if os.getenv("PIPELINE_CALLS_FILE"):                   # run_all.py adds these up for its summary
        with open(os.environ["PIPELINE_CALLS_FILE"], "a") as f:
            f.write(json.dumps(CALLS) + "\n")
    if errors:
        print(pd.DataFrame(errors, columns=["api", "symbol", "error"]).head(10).to_string(index=False))
    if MODE == "online" and not articles:
        raise RuntimeError("No articles fetched in online mode - keeping previous outputs untouched")

### Filter & score

In [ ]:
if MODE == "offline":
    news = pd.read_csv(NEWS_CSV)                       # already FinBERT-labelled (non-neutral only)
else:
    news = pd.DataFrame(articles, columns=["symbol", "date", "headline", "summary", "source"])
news["headline"] = news["headline"].fillna("").astype(str).str.strip()
news["summary"] = news["summary"].fillna("").astype(str).str.strip()
news = news[news["summary"] != ""].drop_duplicates(subset=["symbol", "headline", "summary"])
news = news[[is_relevant(s, f"{h} {m}") for s, h, m in zip(news["symbol"], news["headline"], news["summary"])]].copy()
news["date"] = pd.to_datetime(news["date"], utc=True, format="mixed", errors="coerce")

if MODE != "offline" and len(news):
    from transformers import pipeline
    finbert = pipeline("sentiment-analysis", model="yiyanghkust/finbert-tone", framework="pt")
    texts = [f"{h}. {m}".strip() for h, m in zip(news["headline"], news["summary"])]
    results = finbert(texts, truncation=True, max_length=512, batch_size=16)
    cutoff = {"positive": POSITIVE_CUTOFF, "negative": NEGATIVE_CUTOFF}
    news["sentiment_label"] = [r["label"].lower() if r["label"].lower() in cutoff and r["score"] >= cutoff[r["label"].lower()]
                               else "neutral" for r in results]
elif MODE != "offline":
    news["sentiment_label"] = pd.Series(dtype=str)
print(f"Relevant articles: {len(news)}")
print(news["sentiment_label"].value_counts().to_string())

### Weighted sentiment & save

Negative articles count double. Score = (positive − 2×negative) / (positive + 2×negative + 5) × 10, so stocks with few articles stay
near 0; stocks without any relevant non-neutral article get exactly 0.

In [ ]:
counts = (news.pivot_table(index="symbol", columns="sentiment_label", aggfunc="size")
          .reindex(index=SYMBOLS, columns=["positive", "negative"]).fillna(0))
pos, neg2 = counts["positive"], counts["negative"] * 2
weighted = pd.DataFrame({"Symbol": counts.index, "SentimentScore": ((pos - neg2) / (pos + neg2 + PRIOR_ARTICLES) * 10).round(2).values,
                         "Positive": counts["positive"].astype(int).values, "Negative": counts["negative"].astype(int).values})
kept = news.loc[news["sentiment_label"] != "neutral", ["symbol", "date", "headline", "summary", "source", "sentiment_label"]]

if MODE == "sample":
    kept.to_csv(os.path.join(CACHE_DIR, "sample_news.csv"), index=False)
    weighted.to_csv(os.path.join(CACHE_DIR, "sample_weighted_sentiment.csv"), index=False)
    print("Sample mode: wrote Reports/cache/sample_*.csv only (production outputs untouched)")
else:
    kept.to_csv(NEWS_CSV, index=False)
    weighted.to_csv(SCORE_CSV, index=False)
    if MODE == "online":
        hist = weighted.assign(as_of=str(END))
        if os.path.exists(HISTORY_CSV):
            hist = pd.concat([pd.read_csv(HISTORY_CSV), hist]).drop_duplicates(["as_of", "Symbol"], keep="last")
        hist.to_csv(HISTORY_CSV, index=False)
    print(f"✅ Saved sentiment for {len(weighted)} symbols ({(weighted['SentimentScore'] != 0).sum()} non-zero), {len(kept)} articles")
weighted.sort_values("SentimentScore", ascending=False).head(10)